In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOG = "gwd_de"

# Source tables from silver
source_line_items = f"{CATALOG}.silver.invoice_line_items"
source_invoices = f"{CATALOG}.silver.invoices"
source_payments = f"{CATALOG}.silver.payments"

# Dimension tables from gold
dim_customers = f"{CATALOG}.gold.dim_customers"
dim_products = f"{CATALOG}.gold.dim_products"
dim_regions = f"{CATALOG}.gold.dim_regions"

# Target fact tables in gold
fact_line_items_table = f"{CATALOG}.gold.fact_invoice_line_items"
fact_payments_table = f"{CATALOG}.gold.fact_payments"

In [0]:
# Read from silver
df_line_items = spark.table(source_line_items)
df_invoices = spark.table(source_invoices)

# Join line items with invoices to get customer and date context
fact_line_items = df_line_items.alias("li") \
    .join(df_invoices.alias("inv"), "invoice_id", "left") \
    .select(
        F.col("li.invoice_line_id"),
        F.col("li.invoice_id"),
        F.col("li.product_id"),
        F.col("inv.customer_id"),
        F.col("inv.region"),
        F.col("li.quantity"),
        F.col("li.unit_price"),
        F.col("li.discount"),
        F.col("li.line_total"),
        F.col("inv.invoice_date"),
        F.col("inv.currency"),
        F.col("inv.invoice_status")
    )

# Write to gold
fact_line_items.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(fact_line_items_table)


In [0]:
# Read from silver
df_payments = spark.table(source_payments)
df_invoices = spark.table(source_invoices)

# Join payments with invoices to get customer context
fact_payments = df_payments.alias("pay") \
    .join(df_invoices.alias("inv"), "invoice_id", "left") \
    .select(
        F.col("pay.payment_id"),
        F.col("pay.invoice_id"),
        F.col("inv.customer_id"),
        F.col("inv.region"),
        F.col("pay.payment_date"),
        F.col("pay.payment_amount"),
        F.col("pay.payment_method")
    )

# Write to gold
fact_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(fact_payments_table)
